In [20]:
# - -  - - - - - - - - - Part 1  - - - - - - - - - - - - - - - - - - - - - 
import numbers


class matrix:


    def __init__(self, n, m=None):
        # matrix(2, 3) -> 2x3 matrix of zeros
        # matrix([[1, 2], [3, 4]]) -> matrix from values
        if m is not None:
            if n <= 0 or m <= 0:
                raise ValueError("Matrix size must be positive")
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]
        else:
            if not isinstance(n, list) or len(n) == 0:
                raise TypeError("Use matrix(n, m) or matrix(list_of_lists)")

            # matrix has to be rectangular
            if len(n[0]) == 0:
                raise ValueError("Matrix cannot have an empty row")
            if any(len(row) != len(n[0]) for row in n):
                raise ValueError("All rows must have the same number of columns")

            self.data = [row[:] for row in n]

    def __getitem__(self, key):
        # M[i][j] and M[i,j] both work
        if isinstance(key, tuple):
            row, col = key

            # If either part is a slice, return a new matrix.
            if isinstance(row, slice) or isinstance(col, slice):
                rows = self.data[row]
                return matrix([r[col] for r in rows])

            return self.data[row][col]

        return self.data[key]

    def __setitem__(self, key, value):
        # Example: M[0,1] = 99
        if isinstance(key, tuple):
            row, col = key
            self.data[row][col] = value
        else:
            self.data[key] = value[:]

    def assign(self, other):
        # This method gives the value-copying behavior the lab asks for.
        if isinstance(other, matrix):
            other = other.data

        new_matrix = matrix([row[:] for row in other])

        if self.shape() != new_matrix.shape():
            raise ValueError("Matrix sizes must match")

        self.data = new_matrix.data


# ------------------------- part 2 --------------------- - - - - - - - - - - - - - - 
    def shape(self):
        # memory: shape = (rows, columns)
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        # memory: rows become columns
        return matrix([[self.data[r][c] for r in range(self.shape()[0])]
                       for c in range(self.shape()[1])])

    def row(self, n):
        return matrix([self.data[n][:]])

    def column(self, n):
        return matrix([[self.data[r][n]] for r in range(self.shape()[0])])

    def to_list(self):
        # turns matrix back into a normal Python list of lists
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        # n_0:n_1 = columns, m_0:m_1 = rows.
        # python slices stop before the ending number
        return self[m_0:m_1, n_0:n_1]

# ---------------------------------- part 4 ----------------------------------
    def scalarmul(self, c):
        # scalar multiplication = multiply EVERY element by c
        return matrix([[c * value for value in row] for row in self.data])

    def add(self, other):
        # same position + same position
        self._check_same_size(other)
        return matrix([[self.data[r][c] + other.data[r][c]
                        for c in range(self.shape()[1])]
                       for r in range(self.shape()[0])])

    def sub(self, other):
        # same position - same position
        self._check_same_size(other)
        return matrix([[self.data[r][c] - other.data[r][c]
                        for c in range(self.shape()[1])]
                       for r in range(self.shape()[0])])

    def mat_mult(self, other):
        # memory: ROW x COLUMN, then add
        if not isinstance(other, matrix) or self.shape()[1] != other.shape()[0]:
            raise ValueError("Incompatible matrix sizes for multiplication")

        result = []

        for r in range(self.shape()[0]):
            new_row = []
            for c in range(other.shape()[1]):
                total = 0
                for k in range(self.shape()[1]):
                    total += self.data[r][k] * other.data[k][c]
                new_row.append(total)
            result.append(new_row)

        return matrix(result)

    def element_mult(self, other):
        # memory: SAME BOX x SAME BOX
        self._check_same_size(other)
        return matrix([[self.data[r][c] * other.data[r][c]
                        for c in range(self.shape()[1])]
                       for r in range(self.shape()[0])])

    def equals(self, other):
        # true only when the size and every value match
        return isinstance(other, matrix) and self.data == other.data

    def _check_same_size(self, other):
        if not isinstance(other, matrix) or self.shape() != other.shape():
            raise ValueError("Matrices must have the same size")

# --------------------------------- part 5 --------------------------------------
    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        # M * 2 -> scalar multiplication
        # M * N -> matrix multiplication
        if isinstance(other, numbers.Number):
            return self.scalarmul(other)
        if isinstance(other, matrix):
            return self.mat_mult(other)
        return NotImplemented

    def __rmul__(self, other):
        # 2 * M -> scalar multiplication
        return self.scalarmul(other)

    def __eq__(self, other):
        return self.equals(other)

    def __repr__(self):
        return f"matrix({self.data!r})"


In [21]:
# part 1 test
# create with size
A = matrix(2, 3)
print(A)
assert A.to_list() == [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]

# creating with values
B = matrix([[1, 2, 3], [4, 5, 6]])
assert B[0][1] == 2       # M[i][j]
assert B[1, 2] == 6       # M[i,j]

# changing values
B[0][1] = 20
B[1, 2] = 60
assert B[0][1] == 20
assert B[1, 2] == 60

# bad matrix: rows are different lengths
try:
    matrix([[1, 2], [3]])
    raise AssertionError("Ragged matrix should fail")
except ValueError:
    print("Rectangular check: PASS")

# copy values with assign()
C = matrix([[1, 2], [3, 4]])
D = matrix([[5, 6], [7, 8]])
C.assign(D)
assert C == D

C.assign([[9, 10], [11, 12]])
assert C.to_list() == [[9, 10], [11, 12]]

print("Part 1: PASS")

matrix([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
Rectangular check: PASS
Part 1: PASS


In [22]:
# part 2 test 
M = matrix([[1, 2, 3], [4, 5, 6]])

assert M.shape() == (2, 3)
assert M.transpose() == matrix([[1, 4], [2, 5], [3, 6]])
assert M.row(1) == matrix([[4, 5, 6]])
assert M.column(1) == matrix([[2], [5]])
assert M.to_list() == [[1, 2, 3], [4, 5, 6]]
assert M.block(1, 3, 0, 2) == matrix([[2, 3], [5, 6]])

# slicing
assert M[:, 1:3] == matrix([[2, 3], [5, 6]])
assert M[0:1, 0:2] == matrix([[1, 2]])

print("shape:", M.shape())
print("transpose:", M.transpose())
print("row 1:", M.row(1))
print("column 1:", M.column(1))
print("block:", M.block(1, 3, 0, 2))
print("Part 2: PASS")

shape: (2, 3)
transpose: matrix([[1, 4], [2, 5], [3, 6]])
row 1: matrix([[4, 5, 6]])
column 1: matrix([[2], [5]])
block: matrix([[2, 3], [5, 6]])
Part 2: PASS


In [23]:
# part 3 test
def constant(n, m, c):
    # every value is a float
    return matrix([[float(c) for _ in range(m)] for _ in range(n)])


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    # 1s on the main diagonal, 0s everywhere else
    return matrix([[1.0 if r == c else 0.0 for c in range(n)]
                   for r in range(n)])


assert constant(2, 3, 5) == matrix([[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]])
assert zeros(2, 2) == matrix([[0.0, 0.0], [0.0, 0.0]])
assert ones(2, 2) == matrix([[1.0, 1.0], [1.0, 1.0]])
assert eye(3) == matrix([[1.0, 0.0, 0.0],
                         [0.0, 1.0, 0.0],
                         [0.0, 0.0, 1.0]])

print("constant:", constant(2, 3, 5))
print("zeros:", zeros(2, 2))
print("ones:", ones(2, 2))
print("eye:", eye(3))
print("Part 3: PASS")

constant: matrix([[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]])
zeros: matrix([[0.0, 0.0], [0.0, 0.0]])
ones: matrix([[1.0, 1.0], [1.0, 1.0]])
eye: matrix([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
Part 3: PASS


In [24]:
# part 4 testing
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

assert A.scalarmul(2) == matrix([[2, 4], [6, 8]])
assert A.add(B) == matrix([[6, 8], [10, 12]])
assert A.sub(B) == matrix([[-4, -4], [-4, -4]])
assert A.element_mult(B) == matrix([[5, 12], [21, 32]])
assert A.mat_mult(B) == matrix([[19, 22], [43, 50]])
assert A.equals(matrix([[1, 2], [3, 4]]))
assert not A.equals(B)

# dimension checks
try:
    A.add(matrix([[1, 2, 3]]))
    raise AssertionError("Bad add dimensions should fail")
except ValueError:
    print("Add dimension check: PASS")

try:
    A.mat_mult(matrix([[1, 2, 3]]))
    raise AssertionError("Bad multiply dimensions should fail")
except ValueError:
    print("Multiply dimension check: PASS")

print("A * B:", A.mat_mult(B))
print("A element_mult B:", A.element_mult(B))
print("Part 4: PASS")

Add dimension check: PASS
Multiply dimension check: PASS
A * B: matrix([[19, 22], [43, 50]])
A element_mult B: matrix([[5, 12], [21, 32]])
Part 4: PASS


In [25]:
# part 5 testing 
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

assert 2 * A == A.scalarmul(2)
assert A * 2 == A.scalarmul(2)
assert A + B == A.add(B)
assert A - B == A.sub(B)
assert A * B == A.mat_mult(B)
assert A == matrix([[1, 2], [3, 4]])
assert A != B

print("2*A:", 2 * A)
print("A*2:", A * 2)
print("A+B:", A + B)
print("A-B:", A - B)
print("A*B:", A * B)
print("A==A:", A == A)
print("Part 5: PASS")

2*A: matrix([[2, 4], [6, 8]])
A*2: matrix([[2, 4], [6, 8]])
A+B: matrix([[6, 8], [10, 12]])
A-B: matrix([[-4, -4], [-4, -4]])
A*B: matrix([[19, 22], [43, 50]])
A==A: True
Part 5: PASS


In [27]:
# part 6 testing 
Matrix = matrix

A = Matrix([[1, 2], [3, 4]])
B = Matrix([[2, 0], [1, 2]])
C = Matrix([[0, 1], [1, 0]])
I = Matrix([[1, 0], [0, 1]])

# 1 (AB)C = A(BC)
assert (A * B) * C == A * (B * C)
print("(AB)C = A(BC): PASS")

# 2 A(B+C) = AB+AC
assert A * (B + C) == (A * B) + (A * C)
print("A(B+C) = AB+AC: PASS")

# 3 AB != BA
assert (A * B) != (B * A)
print("AB != BA: PASS")
print("AB =", A * B)
print("BA =", B * A)

# 4 AI = A
assert A * I == A
print("AI = A: PASS")

(AB)C = A(BC): PASS
A(B+C) = AB+AC: PASS
AB != BA: PASS
AB = matrix([[4, 4], [10, 8]])
BA = matrix([[2, 4], [7, 10]])
AI = A: PASS
